# 🤖 Automated Hyperparameter Tuning & Model Selection
### Optuna + MLflow + Unity Catalog — 100% Free on Databricks Free Edition

**Dataset:** Wine Quality (sklearn built-in)
**Task:** Multiclass Classification — `low / medium / high` quality
**Stack:** Optuna · MLflow · Unity Catalog · pandas · sklearn · LightGBM · XGBoost

> ✅ No paid Databricks features used. Runs entirely on Free Edition serverless compute.

---
### 🗂️ Unity Catalog Structure
```
main/
└── wine_project/
    ├── wine_features        ← Feature table (catalog-managed Delta)
    ├── wine_quality_train   ← Training labels + lookup key
    ├── wine_quality_test    ← Test labels + lookup key
    └── wine_predictions     ← Batch inference output
```

---
### 📋 Sections
| # | Section | Description |
|---|---|---|
| 1 | Setup | Install libs, configure Unity Catalog |
| 2 | Feature Engineering | Build & store features in UC Delta table |
| 3 | Model Search | Optuna tunes 4 algorithms, MLflow logs all trials |
| 4 | Results Analysis | Compare trials, evaluate best model |
| 5 | Model Registration | Register best model to UC Model Registry |
| 6 | Batch Inference | Load registered model, score & save predictions |

## 📦 Section 1 — Setup

In [0]:
%pip install optuna lightgbm xgboost
%restart_python

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost
import optuna
import warnings
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import lightgbm as lgb
import xgboost as xgb
 
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
 
print(f"✅ Libraries loaded | MLflow: {mlflow.__version__} | Optuna: {optuna.__version__}")

### 1.1 — Unity Catalog Constants

In [0]:
CATALOG       = "main"
SCHEMA        = "wine_project"
MODEL_NAME    = f"{CATALOG}.{SCHEMA}.wine_quality_classifier"
EXPERIMENT    = "/Users/d.raimondi2000@gmail.com/wine_quality_optuna"   # ← update with your Databricks username
 
# Set Unity Catalog as the default
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
 
mlflow.set_registry_uri("databricks-uc")           # Point MLflow to Unity Catalog
mlflow.set_experiment(EXPERIMENT)
 
print(f"✅ Using catalog: {CATALOG}.{SCHEMA}")
print(f"   Model registry: {MODEL_NAME}")

---
## 🛠️ Section 2 — Feature Engineering & Unity Catalog Storage

### 2.1 — Load Raw Dataset

In [0]:
wine = load_wine()
raw_df = pd.DataFrame(wine.data, columns=wine.feature_names)
 
# Rename the awkward OD column for easier handling
raw_df = raw_df.rename(columns={"od280/od315_of_diluted_wines": "od280_od315"})
 
# Create a unique primary key — required for Feature Store lookup
raw_df["wine_id"] = raw_df.index.astype(int)
 
# Target: map numeric class to readable label
label_map = {0: "low", 1: "medium", 2: "high"}
raw_df["quality"] = pd.Series(wine.target).map(label_map)
 
print(f"Raw dataset: {raw_df.shape[0]} rows × {raw_df.shape[1]} cols")
print(f"Class distribution:\n{raw_df['quality'].value_counts().to_string()}")
display(raw_df.head(5))

### 2.2 — Engineer Features

In [0]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    All feature transformations live here.
    Keeping them in one function makes it easy to reuse at inference time.
 
    Exact sklearn Wine dataset columns:
    alcohol, malic_acid, ash, alcalinity_of_ash, magnesium,
    total_phenols, flavanoids, nonflavanoid_phenols, proanthocyanins,
    color_intensity, hue, od280/od315_of_diluted_wines, proline
    """
    feat = df.copy()
 
    # Ratios capturing chemical balance
    feat["phenols_ratio"]       = feat["flavanoids"] / (feat["total_phenols"] + 1e-6)
    feat["flavanoid_hue_ratio"] = feat["flavanoids"] / (feat["hue"] + 1e-6)
    feat["alcohol_proline"]     = feat["alcohol"] * feat["proline"]
 
    # Log-transform right-skewed features
    for col in ["magnesium", "proline", "color_intensity"]:
        feat[f"log_{col}"] = np.log1p(feat[col])
 
    # Interaction: acidity × mineral content
    feat["acid_ash_ratio"]      = feat["malic_acid"] / (feat["ash"] + 1e-6)
    feat["od_hue_interaction"]  = feat["od280_od315"] * feat["hue"]
 
    return feat
 
features_df = engineer_features(raw_df)
 
# Separate feature columns (exclude target and id)
feature_cols = [c for c in features_df.columns if c not in ["wine_id", "quality"]]
print(f"✅ Feature columns ({len(feature_cols)}): {feature_cols}")

### 2.3 — Write Feature Table to Unity Catalog

In [0]:
# Feature table: wine_id + all engineered features (NO target column)
feature_table_df = features_df[["wine_id"] + feature_cols]
 
spark.createDataFrame(feature_table_df).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.wine_features")
 
print(f"✅ Feature table saved: {CATALOG}.{SCHEMA}.wine_features")
print(f"   Rows: {feature_table_df.shape[0]} | Feature columns: {len(feature_cols)}")

### 2.4 — Train / Test Split & Save Label Tables

In [0]:
# Labels table: wine_id + target only (kept separate from features — best practice)
labels_df = features_df[["wine_id", "quality"]]
 
train_labels, test_labels = train_test_split(
    labels_df, test_size=0.2, random_state=42, stratify=labels_df["quality"]
)
 
spark.createDataFrame(train_labels).write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.wine_quality_train")
 
spark.createDataFrame(test_labels).write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.wine_quality_test")
 
print(f"✅ Train labels: {len(train_labels)} rows → {CATALOG}.{SCHEMA}.wine_quality_train")
print(f"✅ Test labels : {len(test_labels)} rows → {CATALOG}.{SCHEMA}.wine_quality_test")

### 2.5 — Build Train / Test Matrices by Joining Feature Table

In [0]:
def load_feature_set(label_spark_table: str) -> tuple[pd.DataFrame, pd.Series]:
    """
    Joins label table with feature table on wine_id.
    This mimics what the Feature Store does at training & serving time.
    """
    labels   = spark.table(label_spark_table).toPandas()
    features = spark.table(f"{CATALOG}.{SCHEMA}.wine_features").toPandas()
    merged   = labels.merge(features, on="wine_id", how="inner")
 
    X = merged[feature_cols]
    y = merged["quality"]
    return X, y
 
X_train, y_train = load_feature_set(f"{CATALOG}.{SCHEMA}.wine_quality_train")
X_test,  y_test  = load_feature_set(f"{CATALOG}.{SCHEMA}.wine_quality_test")
 
# Encode string labels to integers (required by LightGBM / XGBoost)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
 
print(f"✅ Train: X={X_train.shape}, y={y_train.shape}")
print(f"✅ Test : X={X_test.shape},  y={y_test.shape}")
print(f"   Label classes: {list(le.classes_)}")

---
## 🔍 Section 3 — Automated Model Search with Optuna

Optuna uses **TPE (Tree-structured Parzen Estimator)** — the same Bayesian algorithm
used under the hood in many AutoML tools. It intelligently narrows the search space
based on previous trial results, making it far more efficient than random or grid search.

### Algorithms in the search:
| Algorithm | Key hyperparameters tuned |
|---|---|
| LightGBM | learning_rate, n_estimators, max_depth, num_leaves, reg terms |
| XGBoost | learning_rate, n_estimators, max_depth, subsample, col_sample |
| Random Forest | n_estimators, max_depth, min_samples_split, max_features |
| Logistic Regression | C, solver, max_iter |

### 3.1 — Define the Objective Function

In [0]:
CV_FOLDS      = 5       # Stratified K-Fold cross-validation
N_TRIALS      = 40      # Total Optuna trials across all algorithms
RANDOM_STATE  = 42
 
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
 
def objective(trial: optuna.Trial) -> float:
    """
    Optuna calls this function for each trial.
    It suggests an algorithm + hyperparameters, trains via CV,
    and logs everything to MLflow.
    Returns the mean CV F1 score (macro) — Optuna maximizes this.
    """
    algorithm = trial.suggest_categorical(
        "algorithm", ["lightgbm", "xgboost", "random_forest", "logistic_regression"]
    )
 
    # ── Build model based on suggested algorithm ──────────────────────────────
    if algorithm == "lightgbm":
        params = {
            "n_estimators"  : trial.suggest_int("lgb_n_estimators", 50, 400),
            "learning_rate" : trial.suggest_float("lgb_learning_rate", 1e-3, 0.3, log=True),
            "max_depth"     : trial.suggest_int("lgb_max_depth", 3, 12),
            "num_leaves"    : trial.suggest_int("lgb_num_leaves", 20, 150),
            "reg_alpha"     : trial.suggest_float("lgb_reg_alpha", 1e-8, 1.0, log=True),
            "reg_lambda"    : trial.suggest_float("lgb_reg_lambda", 1e-8, 1.0, log=True),
            "subsample"     : trial.suggest_float("lgb_subsample", 0.5, 1.0),
            "random_state"  : RANDOM_STATE,
            "verbose"       : -1,
        }
        model = lgb.LGBMClassifier(**params)
 
    elif algorithm == "xgboost":
        params = {
            "n_estimators"        : trial.suggest_int("xgb_n_estimators", 50, 400),
            "learning_rate"       : trial.suggest_float("xgb_learning_rate", 1e-3, 0.3, log=True),
            "max_depth"           : trial.suggest_int("xgb_max_depth", 3, 10),
            "subsample"           : trial.suggest_float("xgb_subsample", 0.5, 1.0),
            "colsample_bytree"    : trial.suggest_float("xgb_colsample_bytree", 0.5, 1.0),
            "reg_alpha"           : trial.suggest_float("xgb_reg_alpha", 1e-8, 1.0, log=True),
            "reg_lambda"          : trial.suggest_float("xgb_reg_lambda", 1e-8, 1.0, log=True),
            "random_state"        : RANDOM_STATE,
            "eval_metric"         : "mlogloss",
            "use_label_encoder"   : False,
        }
        model = xgb.XGBClassifier(**params)
 
    elif algorithm == "random_forest":
        params = {
            "n_estimators"     : trial.suggest_int("rf_n_estimators", 50, 400),
            "max_depth"        : trial.suggest_int("rf_max_depth", 3, 20),
            "min_samples_split": trial.suggest_int("rf_min_samples_split", 2, 15),
            "min_samples_leaf" : trial.suggest_int("rf_min_samples_leaf", 1, 10),
            "max_features"     : trial.suggest_categorical("rf_max_features", ["sqrt", "log2"]),
            "random_state"     : RANDOM_STATE,
        }
        model = RandomForestClassifier(**params)
 
    else:  # logistic_regression
        params = {
            "C"           : trial.suggest_float("lr_C", 1e-4, 100.0, log=True),
            "solver"      : trial.suggest_categorical("lr_solver", ["lbfgs", "saga"]),
            "max_iter"    : trial.suggest_int("lr_max_iter", 200, 2000),
            "random_state": RANDOM_STATE,
        }
        model = LogisticRegression(**params)
 
    # ── Cross-validate ────────────────────────────────────────────────────────
    cv_scores = cross_val_score(
        model, X_train, y_train_enc, cv=cv, scoring="f1_macro", n_jobs=-1
    )
    mean_f1  = cv_scores.mean()
    std_f1   = cv_scores.std()
 
    # ── Log to MLflow (nested run per trial) ──────────────────────────────────
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}_{algorithm}"):
        mlflow.log_param("algorithm", algorithm)
        mlflow.log_params({k: v for k, v in params.items() if k != "random_state"})
        mlflow.log_metric("cv_f1_macro_mean", mean_f1)
        mlflow.log_metric("cv_f1_macro_std",  std_f1)
        mlflow.log_metric("trial_number",     trial.number)
 
    return mean_f1

### 3.2 — Run the Optuna Study

In [0]:
with mlflow.start_run(run_name="optuna_model_search") as parent_run:
 
    mlflow.log_param("n_trials",   N_TRIALS)
    mlflow.log_param("cv_folds",   CV_FOLDS)
    mlflow.log_param("dataset",    "wine_quality")
    mlflow.log_param("n_features", len(feature_cols))
 
    study = optuna.create_study(
        direction   = "maximize",
        sampler     = optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner      = optuna.pruners.MedianPruner(n_startup_trials=5),
        study_name  = "wine_quality_model_search",
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
 
    # Log best result to parent run
    mlflow.log_metric("best_cv_f1_macro", study.best_value)
    mlflow.log_param("best_algorithm",    study.best_params["algorithm"])
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
 
    PARENT_RUN_ID = parent_run.info.run_id
 
print("\n" + "=" * 55)
print("✅  OPTUNA STUDY COMPLETE")
print("=" * 55)
print(f"  Total trials      : {len(study.trials)}")
print(f"  Best CV F1 (macro): {study.best_value:.4f}")
print(f"  Best algorithm    : {study.best_params['algorithm']}")
print(f"  Best params       : {study.best_params}")
print(f"  Parent MLflow run : {PARENT_RUN_ID}")
 

---
## 📊 Section 4 — Results Analysis

### 4.1 — Trial Comparison Table

In [0]:
trials_df = study.trials_dataframe()
trials_df = trials_df[["number", "value", "params_algorithm", "duration"]].copy()
trials_df.columns = ["trial", "cv_f1_macro", "algorithm", "duration"]
trials_df = trials_df.sort_values("cv_f1_macro", ascending=False).reset_index(drop=True)
trials_df["rank"] = trials_df.index + 1
 
print(f"Top 10 trials:")
display(trials_df.head(10))

### 4.2 — Algorithm Performance Summary


In [0]:
algo_summary = (
    trials_df.groupby("algorithm")["cv_f1_macro"]
    .agg(["mean", "max", "std", "count"])
    .round(4)
    .sort_values("mean", ascending=False)
    .reset_index()
)
algo_summary.columns = ["algorithm", "mean_f1", "best_f1", "std_f1", "n_trials"]
print("Algorithm performance across all trials:")
display(algo_summary)

### 4.3 — Retrain Best Model on Full Training Set & Evaluate on Test Set

In [0]:
best_params = study.best_params.copy()
best_algo   = best_params.pop("algorithm")
 
print(f"🏆 Best algorithm : {best_algo}")
print(f"   Best params    : {best_params}\n")
 
# Rebuild the best model with the full training set
if best_algo == "lightgbm":
    lgb_params = {k.replace("lgb_", ""): v for k, v in best_params.items()}
    lgb_params["random_state"] = RANDOM_STATE
    lgb_params["verbose"] = -1
    best_model = lgb.LGBMClassifier(**lgb_params)
 
elif best_algo == "xgboost":
    xgb_params = {k.replace("xgb_", ""): v for k, v in best_params.items()}
    xgb_params["random_state"] = RANDOM_STATE
    xgb_params["eval_metric"] = "mlogloss"
    xgb_params["use_label_encoder"] = False
    best_model = xgb.XGBClassifier(**xgb_params)
 
elif best_algo == "random_forest":
    rf_params = {k.replace("rf_", ""): v for k, v in best_params.items()}
    rf_params["random_state"] = RANDOM_STATE
    best_model = RandomForestClassifier(**rf_params)
 
else:
    lr_params = {k.replace("lr_", ""): v for k, v in best_params.items()}
    lr_params["random_state"] = RANDOM_STATE
    best_model = LogisticRegression(**lr_params)
 
# Fit on full training data
best_model.fit(X_train, y_train_enc)
 
# Evaluate on test set
y_pred_enc = best_model.predict(X_test)
y_pred     = le.inverse_transform(y_pred_enc)
test_f1    = f1_score(y_test_enc, y_pred_enc, average="macro")
 
print(f"📊 Test Set Results")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\n✅ Test F1 Macro: {test_f1:.4f}")

---
## 🗂️ Section 5 — Register Best Model to Unity Catalog

### 5.1 — Log & Register the Best Mode

In [0]:
with mlflow.start_run(run_name="best_model_registration") as reg_run:
 
    # Log params & metrics
    mlflow.log_param("algorithm",     best_algo)
    mlflow.log_param("feature_table", f"{CATALOG}.{SCHEMA}.wine_features")
    mlflow.log_params(best_params)
    mlflow.log_metric("test_f1_macro",    test_f1)
    mlflow.log_metric("best_cv_f1_macro", study.best_value)
 
    # Build signature & input example — required for Unity Catalog registration
    from mlflow.models.signature import infer_signature
 
    X_sample   = X_train.iloc[:5]                    # Small representative sample
    y_sample   = best_model.predict(X_sample)        # Predicted output for signature
    signature  = infer_signature(X_sample, y_sample) # Captures input/output schema
 
    # Shared kwargs for all log_model calls
    log_kwargs = dict(
        registered_model_name = MODEL_NAME,
        input_example         = X_sample,
        signature             = signature,
    )
 
    # Log & register model — use the right flavour per algorithm
    if best_algo == "lightgbm":
        mlflow.lightgbm.log_model(best_model, artifact_path="model", **log_kwargs)
    elif best_algo == "xgboost":
        mlflow.xgboost.log_model(best_model,  artifact_path="model", **log_kwargs)
    else:
        mlflow.sklearn.log_model(best_model,   artifact_path="model", **log_kwargs)
 
    # Log label encoder as artifact (needed at inference time)
    import pickle, os, tempfile
    with tempfile.TemporaryDirectory() as tmp:
        le_path = os.path.join(tmp, "label_encoder.pkl")
        with open(le_path, "wb") as f:
            pickle.dump(le, f)
        mlflow.log_artifact(le_path, artifact_path="label_encoder")
 
    REG_RUN_ID = reg_run.info.run_id
 
print(f"✅ Model registered to Unity Catalog: {MODEL_NAME}")
print(f"   MLflow run ID: {REG_RUN_ID}")

### 5.2 — Set Alias & Add Description

In [0]:
from mlflow import MlflowClient
 
client = MlflowClient()
 
# UC doesn't support .latest_versions — use search_model_versions instead
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
version  = sorted(versions, key=lambda v: int(v.version))[-1].version
 
# Set "champion" alias (UC uses aliases, not stages)
client.set_registered_model_alias(MODEL_NAME, "champion", version)
 
# Annotate with description
client.update_model_version(
    name        = MODEL_NAME,
    version     = version,
    description = (
        f"Best Optuna trial. Algorithm: {best_algo}. "
        f"CV F1: {study.best_value:.4f} | Test F1: {test_f1:.4f}. "
        f"Features: {CATALOG}.{SCHEMA}.wine_features"
    ),
)
 
print(f"✅ Version {version} registered with alias 'champion'")
print(f"   Description set with metrics and feature table reference")

---
## 🔮 Section 6 — Batch Inference

### 6.1 — Load Champion Model from Unity Catalog

In [0]:
import mlflow.pyfunc
 
# Load by alias — always gets the current "champion" without hardcoding a version
champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
print(f"✅ Loaded model: {MODEL_NAME}@champion (version {version})")

### 6.2 — Join Test Labels with Feature Table & Predict

In [0]:
# Always read features from the catalog — single source of truth
X_infer, y_true = load_feature_set(f"{CATALOG}.{SCHEMA}.wine_quality_test")
predictions_enc  = champion_model.predict(X_infer)
predictions      = le.inverse_transform(predictions_enc)
 
# Build results DataFrame
test_labels_df = spark.table(f"{CATALOG}.{SCHEMA}.wine_quality_test").toPandas()
results_df = test_labels_df.copy()
results_df["predicted_quality"] = predictions
results_df["correct"]           = results_df["quality"] == results_df["predicted_quality"]
 
accuracy = results_df["correct"].mean()
print(f"✅ Batch inference complete — {len(results_df)} predictions")
print(f"   Overall accuracy: {accuracy:.4f}")
display(results_df.head(20))

### 6.3 — Save Predictions to Unity Catalog

In [0]:
spark.createDataFrame(results_df) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.wine_predictions")
 
print(f"✅ Predictions saved to: {CATALOG}.{SCHEMA}.wine_predictions")

---
## ✅ Project Summary

| Step | What was done |
|---|---|
| 1. Setup | Configured Unity Catalog (`main.wine_project`), pointed MLflow to UC registry |
| 2. Features | Engineered features saved to `wine_features` Delta table in UC |
| 3. Model Search | Optuna TPE search across 4 algorithms, 40 trials, all logged to MLflow |
| 4. Analysis | Compared all trials, retrained best model, evaluated on held-out test set |
| 5. Registration | Best model registered to UC Model Registry with `champion` alias |
| 6. Inference | Champion model loaded from UC, predictions saved back to UC |

### 🔗 Next Steps
- **Increase `N_TRIALS`** for a deeper search (more compute = better results)
- **Add more algorithms**: CatBoost, SVM, ExtraTreesClassifier
- **SHAP explainability**: `import shap; shap.TreeExplainer(best_model)`
- **Schedule retraining**: Use Databricks Workflows to run this notebook on a schedule
- **Model Serving**: Deploy `champion` model via Databricks Model Serving (paid feature)